# 🖥️ GPUBid: Agentic GPU Marketplace

Buyer agents represent AI/ML jobs. Seller agents represent GPU capacity slots. They negotiate over rounds via structured offer tuples plus free-form reasoning. A central engine clears agreed deals.

This notebook is the canonical artifact: every cell produces a visual or interactive output. Scroll top-to-bottom to follow the story.

## Setup

Run once when you open the notebook. In Colab this also enables custom widgets so `ipywidgets` work inline.

In [ ]:
# Setup. Behaviour by environment:
#   - Colab: deletes any old clone, re-clones from GitHub, AND evicts any cached
#     `gpubid.*` modules from `sys.modules` so re-running this cell after a code
#     update genuinely picks up the new classes (Python doesn't auto-reload).
#     Comment out the `rm -rf` line if you have local edits to keep.
#   - Local: imports from `../src` (notebook is in `notebooks/`).
import os, sys, subprocess

GITHUB_URL = 'https://github.com/meghna810/GPUBid.git'

if 'google.colab' in sys.modules:
    !rm -rf /content/GPUBid 2>/dev/null
    !git clone -q $GITHUB_URL /content/GPUBid
    # google-genai is the official Google AI SDK for Gemini (the 2024+ replacement
    # for google-generativeai). Required for the Gemini-backed runs in cells 6.7+.
    !pip install --quiet pydantic numpy pulp plotly matplotlib ipywidgets anthropic openai google-genai tenacity
    REPO_ROOT = '/content/GPUBid'
    from google.colab import output
    output.enable_custom_widget_manager()
else:
    here = os.path.abspath('.')
    parent = os.path.abspath('..')
    candidates = [parent, here, '/content/GPUBid', '/content/drive/MyDrive/GPUBid']
    REPO_ROOT = next((c for c in candidates if os.path.isdir(os.path.join(c, 'src', 'gpubid'))), None)
    if REPO_ROOT is None:
        raise RuntimeError(
            "Couldn't find the gpubid package. Make sure you're running from the repo root, "
            "or run this notebook in Colab where it auto-clones from GitHub."
        )

# Evict any previously-imported gpubid modules so we don't pick up stale classes
# from an earlier run. Without this, re-running the setup cell pulls fresh files
# to disk but Python keeps the OLD classes in memory and you get cryptic
# AttributeErrors like "RoundSnapshot has no attribute 'actions'".
# (For the deepest reset — including any variables that hold old class instances —
# use Runtime -> Restart runtime in Colab.)
for cached_name in [n for n in list(sys.modules) if n == 'gpubid' or n.startswith('gpubid.')]:
    del sys.modules[cached_name]

src_path = os.path.join(REPO_ROOT, 'src')
if src_path in sys.path:
    sys.path.remove(src_path)
sys.path.insert(0, src_path)

import gpubid

def _git(args):
    try:
        return subprocess.run(['git', '-C', REPO_ROOT] + args,
                              capture_output=True, text=True, check=True).stdout.strip()
    except Exception:
        return None

commit_hash = _git(['rev-parse', '--short', 'HEAD']) or '?'
commit_subj = _git(['log', '-1', '--format=%s']) or '(no git info)'
print(f'gpubid v{gpubid.__version__}  ·  commit {commit_hash}: {commit_subj}')
print(f'  repo: {REPO_ROOT}')

## Mode + API key

Two modes; v0.3 is LLM-only (the deterministic "fast" mode is gone — every negotiation is real LLM).

- **Live** (default when a key is loaded) — runs a fresh LLM negotiation now. ~30-60 seconds per market. Costs API credits proportional to market size.
- **Preset** — replays a previously-baked LLM negotiation from `data/presets/*.json`. No API cost, instant playback. Use this for the live class showcase — classmates clicking the URL pay nothing, the demo is reproducible.

**Three providers are now supported** — the key prefix tells the framework which to use:

| Prefix | Provider | Default model | Where to get a key |
|---|---|---|---|
| `sk-ant-…` | Anthropic Claude | `claude-haiku-4-5` | console.anthropic.com |
| `sk-…` | OpenAI | `gpt-4o-mini` | platform.openai.com |
| `AIza…` | Google Gemini | `gemini-2.5-flash` | aistudio.google.com |

API keys come from (in priority order): Colab Secrets (`ANTHROPIC_API_KEY` / `OPENAI_API_KEY` / `GEMINI_API_KEY`, with notebook access enabled), the password field below, or env vars. The provider auto-detects from the prefix.

In [ ]:
import ipywidgets as w
from gpubid.experiments.bake_presets import list_presets
from pathlib import Path

# Try to load an API key from Colab Secrets first. Order: Anthropic → OpenAI → Gemini.
# (Whichever fires first wins; provider is auto-detected from the key prefix.)
api_key_default = ''
secret_loaded_name = None
if 'google.colab' in sys.modules:
    try:
        from google.colab import userdata
        for secret_name in ['ANTHROPIC_API_KEY', 'OPENAI_API_KEY', 'GEMINI_API_KEY']:
            try:
                value = userdata.get(secret_name)
                if value:
                    api_key_default = value
                    secret_loaded_name = secret_name
                    break
            except Exception:
                continue
    except ImportError:
        pass

# Local fallback: env vars (in case running outside Colab).
if not api_key_default:
    for env in ['ANTHROPIC_API_KEY', 'OPENAI_API_KEY', 'GEMINI_API_KEY', 'GOOGLE_API_KEY']:
        v = os.environ.get(env)
        if v:
            api_key_default = v
            secret_loaded_name = env
            break

if secret_loaded_name:
    print(f'✓ Loaded {secret_loaded_name} — defaulting to live mode.')
else:
    print('No API key found in Colab Secrets or env. Paste one below or pick preset mode.')

# Pick default mode: live if a key is loaded, preset otherwise.
preset_paths = list_presets(Path(REPO_ROOT) / 'data' / 'presets')
mode_options = [
    ('live (real LLM — runs negotiation now)', 'live'),
    ('preset (cached LLM playback — no API cost)', 'preset'),
]
default_mode = 'live' if api_key_default else 'preset'
if default_mode == 'preset' and not preset_paths:
    default_mode = 'live'

mode_select = w.Dropdown(
    options=mode_options, value=default_mode,
    description='Mode',
    style={'description_width': 'initial'},
)
api_key_input = w.Password(
    value=api_key_default,
    placeholder='sk-ant-... or sk-... or AIza... (or set in Colab Secrets)',
    description='API key',
    style={'description_width': 'initial'},
)
preset_select = w.Dropdown(
    options=[(p.stem, str(p)) for p in preset_paths] or [('(no presets baked yet)', '')],
    description='Preset',
    style={'description_width': 'initial'},
)
w.VBox([mode_select, api_key_input, preset_select])

## 0.5 The kind of input the buyer agent receives

In the v0.3 design, buyers don\'t arrive with a perfectly structured 6-dim spec. They arrive with a CEO/CTO message in plain English. The **buyer agent\'s job** is to translate that into a structured profile — qty, GPU type preferences, time window, urgency band, max willingness-to-pay — that goes into negotiation.

Here are 12 sample inputs the buyer agent would translate. The market generator currently uses synthetic structured profiles (the buyer cards in cell 8); the translate step is wired up in `gpubid.agents.buyer_agent.BuyerAgent.translate()` and runs in live mode once API keys are provided.

In [ ]:
from gpubid.domain.requirements import REQUIREMENT_LIBRARY
from IPython.display import HTML, display

rows = []
for r in REQUIREMENT_LIBRARY[:6]:  # show first 6 to keep the cell tidy
    rows.append(
        f'<div style="background:#fff;border-left:4px solid #0369a1;border:1px solid #e5e7eb;'
        f'border-left-width:4px;border-radius:6px;padding:10px 12px;font-family:-apple-system, sans-serif;'
        f'box-shadow:0 1px 2px rgba(0,0,0,0.04);">'
        f'<div style="font-size:11px;color:#6b7280;text-transform:uppercase;letter-spacing:0.05em;">'
        f'{r.persona}'
        f'</div>'
        f'<div style="font-size:13px;color:#111;margin-top:6px;font-style:italic;line-height:1.4;">'
        f'\u201c{r.raw_text}\u201d'
        f'</div>'
        f'<div style="font-size:10px;color:#9ca3af;margin-top:6px;font-family:monospace;">'
        f'{r.requirement_id} · expected: {r.workload_category} · {r.expected_urgency_band} · '
        f'qty {r.expected_qty_range[0]}-{r.expected_qty_range[1]}'
        f'</div>'
        f'</div>'
    )

display(HTML(
    '<div style="display:flex;flex-direction:column;gap:8px;max-width:900px;">' + ''.join(rows) + '</div>'
))
print(f'\n{len(REQUIREMENT_LIBRARY)} total requirements in the library. '
      'BuyerAgent.translate() converts each into a (BuyerPublicProfile, BuyerPrivateProfile) pair.')

## 1. Pick a market

Move the sliders to choose how many buyers and sellers, the supply regime (`tight` = constrained, `slack` = generous), and a random seed.

_Note: in **preset** mode these sliders are ignored — the market comes from the preset file._

In [ ]:
buyers_slider  = w.IntSlider(value=8, min=3, max=12, description='# Buyers')
sellers_slider = w.IntSlider(value=4, min=2, max=6,  description='# Sellers')
regime_select  = w.Dropdown(options=['tight', 'slack'], value='tight', description='Regime')
seed_slider    = w.IntSlider(value=42, min=0, max=999, description='Seed')
rounds_slider  = w.IntSlider(value=5, min=1, max=10,  description='Max rounds')
w.VBox([buyers_slider, sellers_slider, regime_select, seed_slider, rounds_slider])

## 2. Render the market

Each buyer card shows the structured profile (qty, GPU prefs, time window, urgency) — but in v0.3 these are **derived from a CEO/CTO requirement** the buyer agent received. Below the cards you'll see one panel per buyer with the original natural-language brief, so you can see what the agent translated.

Each seller card shows their capacity slots; sellers in v0.3 also carry **volume-discount tiers** (publicly advertised), which makes their offers non-directly-comparable — the headline reason agents have value over a planner.

**Private** values (a buyer's max willingness-to-pay, a seller's reserve, urgency score) are visible to *us* in the notebook for inspection — agents only see structured public offers and free-form reasoning.

In [ ]:
from gpubid.market_v3 import generate_market_v3
from gpubid.llm import make_client

# v0.3 market: buyers come from CEO requirements; sellers carry volume-discount
# tiers. If a key is provided AND mode is live, run the LLM-driven translate
# step; otherwise use the deterministic fallback that derives plausible
# structured profiles from each requirement's expected ranges.
_translate_client = None
if mode_select.value == 'live' and api_key_input.value:
    try:
        _translate_client = make_client(api_key_input.value)
        print(f'Using {_translate_client.provider}/{_translate_client.model} for buyer-agent translate step.')
    except Exception as e:
        print(f'Could not init LLM client for translate: {e}')
        print('Falling back to deterministic translate.')

market, enrichment = generate_market_v3(
    n_buyers=buyers_slider.value,
    n_sellers=sellers_slider.value,
    regime=regime_select.value,
    seed=seed_slider.value,
    llm_client=_translate_client,
)
market

In [ ]:
from IPython.display import HTML, display

ceo_cards_html = []
for buyer in market.buyers:
    req = enrichment.buyer_requirements.get(buyer.id)
    pub = enrichment.buyer_public_profiles.get(buyer.id)
    priv = enrichment.buyer_private_profiles.get(buyer.id)
    if req is None:
        continue
    ceo_cards_html.append(
        f'<div style="background:#fff;border-left:4px solid #0369a1;border:1px solid #e5e7eb;'
        f'border-left-width:4px;border-radius:6px;padding:10px 12px;font-family:-apple-system, sans-serif;'
        f'box-shadow:0 1px 2px rgba(0,0,0,0.04);margin-bottom:6px;">'
        f'<div style="display:flex;justify-content:space-between;align-items:baseline;">'
        f'<strong style="font-size:13px;color:#111;">\U0001f4ac {buyer.id} {buyer.label}</strong>'
        f'<span style="font-family:monospace;font-size:10px;color:#9ca3af;">{req.requirement_id}</span>'
        f'</div>'
        f'<div style="font-size:10px;color:#6b7280;margin-top:2px;">{req.persona}</div>'
        f'<div style="font-size:12px;color:#1f2937;margin-top:6px;font-style:italic;line-height:1.4;">'
        f'\u201c{req.raw_text}\u201d'
        f'</div>'
        f'<div style="font-size:10px;color:#9ca3af;margin-top:6px;font-family:monospace;">'
        f'translated to: {pub.workload_category} \u00b7 '
        f'qty {pub.qty_gpus} \u00b7 {pub.duration_hours:.0f}h \u00b7 '
        f'window [{pub.time_window.earliest_start_slot:02d},{pub.time_window.latest_finish_slot:02d}] \u00b7 '
        f'tol={pub.interruption_tolerance} \u00b7 urgency={pub.urgency_band}'
        f'</div>'
        f'<div style="font-size:10px;color:#dc2626;margin-top:2px;font-family:monospace;">'
        f'private: max_wtp=\u0024{priv.max_willingness_to_pay:.2f}/GPU-hr \u00b7 '
        f'urgency_score={priv.urgency_score:.2f} \u00b7 budget=\u0024{priv.budget_remaining_usd:.0f}'
        f'</div>'
        f'</div>'
    )

display(HTML(
    '<div style="margin-top:12px;font-size:12px;font-weight:600;color:#374151;'
    'text-transform:uppercase;letter-spacing:0.05em;margin-bottom:6px;">'
    '\U0001f4ac From the CEO/CTO (the input each buyer agent translated)'
    '</div>'
    + '<div>' + ''.join(ceo_cards_html) + '</div>'
))

# Seller volume-tier policy summary
tier_html = []
for seller in market.sellers:
    pol = enrichment.seller_volume_policies.get(seller.id)
    if pol is None or not pol.tiers:
        tier_summary = 'flat list price (no volume discount advertised)'
    else:
        parts = []
        for tier in pol.tiers:
            parts.append(f'{tier.min_qty_gpus}+ GPUs / {tier.min_duration_hours:.0f}+h \u2192 {tier.discount_pct*100:.0f}% off'  )
        neg = ' (negotiable)' if pol.is_negotiable else ''
        tier_summary = ', '.join(parts) + neg
    tier_html.append(
        f'<div style="font-size:11px;color:#374151;padding:4px 8px;background:#f9fafb;'
        f'border-left:3px solid #16a34a;border-radius:3px;margin-bottom:3px;">'
        f'<strong>{seller.id} {seller.label}</strong>: {tier_summary}'
        f'</div>'
    )

display(HTML(
    '<div style="margin-top:14px;font-size:12px;font-weight:600;color:#374151;'
    'text-transform:uppercase;letter-spacing:0.05em;margin-bottom:6px;">'
    '\U0001f3f7\ufe0f Seller volume-discount policies (publicly advertised)'
    '</div>'
    + ''.join(tier_html)
))

## 3. Watch the agents negotiate

Hit run. Sellers post asks, buyers post bids, and acceptances form deals — animated round-by-round. The deals pane at the bottom flashes green for new agreements.

In [ ]:
from gpubid.viz.trading_floor import animate_negotiation

mode_label = mode_select.value  # 'live' or 'preset' (no fast in v0.3)

if mode_label == 'preset':
    if not preset_select.value:
        raise RuntimeError(
            'No presets in data/presets/. Bake one first with the team API keys:\n'
            '    ANTHROPIC_API_KEY=sk-ant-... PYTHONPATH=src python -m gpubid.experiments.bake_presets all\n'
            'Or switch the Mode dropdown to "live" if you have a key set.'
        )
    final, market, all_snapshots = animate_negotiation(
        mode='preset', preset_path=preset_select.value, step_seconds=1.2,
    )
elif mode_label == 'live':
    if not api_key_input.value:
        raise RuntimeError(
            'Live mode requires an API key.\n'
            'Set ANTHROPIC_API_KEY or OPENAI_API_KEY in Colab Secrets (key icon, left sidebar) '
            'and toggle "Notebook access" on. Or paste a key into the field in the cell above.'
        )
    final, market, all_snapshots = animate_negotiation(
        market, mode='live',
        api_key=api_key_input.value,
        max_rounds=rounds_slider.value, step_seconds=0.4,
    )
else:
    raise RuntimeError(f'Unknown mode: {mode_label}')

n_deals = len(final.all_deals)
total_surplus = sum(d.total_value for d in final.all_deals)
print(f'\n{n_deals} deals struck across {final.round_n} rounds, total transacted: ${total_surplus:.0f}')

## 4. Compare to baselines

How well did the agents do compared to the **welfare-optimal** allocation (VCG, computed by a mixed-integer program — what an oracle planner would assign) and to the **posted-price** baseline (a single take-it-or-leave-it price per GPU type — closer to what current cloud markets give you)?

In [ ]:
from gpubid.benchmark.vcg import solve_vcg
from gpubid.benchmark.vcg_v2 import solve_vcg_tiered
from gpubid.benchmark.posted_price import solve_posted_price
from gpubid.eval.metrics import compute_metrics
from gpubid.viz.charts import baseline_comparison, metric_table
from IPython.display import HTML, display

vcg_result        = solve_vcg(market)
vcg_tiered_result = solve_vcg_tiered(market)
posted_result     = solve_posted_price(market)

agentic_metrics    = compute_metrics(market, list(final.all_deals))
vcg_metrics        = compute_metrics(market, vcg_result.deals)
vcg_tiered_metrics = compute_metrics(market, vcg_tiered_result.deals)
posted_metrics     = compute_metrics(market, posted_result.deals)

# Existing 3-bar chart still shows Agentic vs VCG vs Posted (the headline).
fig = baseline_comparison(
    agentic=agentic_metrics, vcg=vcg_metrics, posted=posted_metrics,
    title=f'Welfare comparison — {market.regime} supply (seed {market.seed})',
)
fig.show()
display(HTML(metric_table(agentic=agentic_metrics, vcg=vcg_metrics, posted=posted_metrics)))

# v0.3 addition: 4-way comparison including tiered VCG.
# On v0.2 markets (no volume-discount policies), VCG-tiered matches legacy VCG;
# on v0.3 markets with non-trivial tiers, the tiered variant materializes
# more allocation options and welfare may differ.
display(HTML(
    f'<div style="font-family:-apple-system,sans-serif;margin-top:12px;font-size:12px;'
    f'background:#f9fafb;border-left:3px solid #2563eb;padding:10px 12px;border-radius:4px;">'
    f'<strong>v0.3 tiered-VCG benchmark:</strong> '
    f'${vcg_tiered_metrics.welfare:.0f} welfare across '
    f'{len(vcg_tiered_result.deals)} assignments. '
    f'<span style="color:#6b7280;">{vcg_tiered_result.linearization_notes}</span>'
    f'</div>'
))

if vcg_metrics.welfare > 0:
    recov  = agentic_metrics.welfare / vcg_metrics.welfare * 100
    pposted = posted_metrics.welfare / vcg_metrics.welfare * 100
    ptiered = vcg_tiered_metrics.welfare / vcg_metrics.welfare * 100
    print(f'\nAgentic recovered {recov:.0f}% of legacy-VCG welfare; '
          f'posted-price recovered {pposted:.0f}%; '
          f'tiered-VCG recovered {ptiered:.0f}% (legacy=100%).')

## 5. Tradeoff sandbox

Two toggles, live: the **concentration cap** and the **round count**. Drag the sliders and watch the welfare-recovery, Gini, and buyer-fill-rate update in real time. Same market as above (cell 2's seed). All in fast mode so it's instantaneous.

_The other two tradeoffs in the proposal — homogeneous-vs-heterogeneous sellers and structured-vs-free-form offers — only show their effect with real LLMs, so they appear in the headline figures (cell 7) and in dedicated presets._

In [ ]:
from gpubid.viz.trading_floor import collect_snapshots

vcg_for_sandbox = solve_vcg(market)
posted_for_sandbox = solve_posted_price(market)
posted_metrics_sb = compute_metrics(market, posted_for_sandbox.deals)

@w.interact(
    cap_pct=w.FloatSlider(value=0.0, min=0.0, max=0.25, step=0.025, description='Cap %', readout_format='.1%'),
    max_rounds=w.IntSlider(value=5, min=1, max=10, description='Max rounds'),
)
def explore_tradeoffs(cap_pct, max_rounds):
    cap = cap_pct if cap_pct > 0 else None
    snaps = collect_snapshots(market, max_rounds=max_rounds, concentration_cap_pct=cap)
    deals = list(snaps[-1].all_deals)
    m = compute_metrics(market, deals)
    recov = m.welfare / vcg_for_sandbox.welfare * 100 if vcg_for_sandbox.welfare > 0 else 0
    posted_recov = posted_metrics_sb.welfare / vcg_for_sandbox.welfare * 100 if vcg_for_sandbox.welfare > 0 else 0

    def stat(label, value, hint=''):
        return (f'<div style="flex:1;text-align:center;background:#fff;border:1px solid #e5e7eb;'
                f'border-radius:8px;padding:10px;">'
                f'<div style="font-size:11px;color:#6b7280;text-transform:uppercase;">{label}</div>'
                f'<div style="font-size:22px;font-weight:600;color:#111;">{value}</div>'
                f'<div style="font-size:10px;color:#9ca3af;">{hint}</div>'
                f'</div>')

    html = (f'<div style="display:flex;gap:8px;font-family:-apple-system, sans-serif;margin:8px 0;">'
            f'{stat("Recovery", f"{recov:.0f}%", f"vs posted {posted_recov:.0f}%")}'
            f'{stat("Buyers filled", f"{m.n_buyers_filled}/{len(market.buyers)}")}'
            f'{stat("Avg price", f"${m.avg_clearing_price:.2f}")}'
            f'{stat("Gini", f"{m.gini_buyer_welfare:.3f}", "buyer welfare inequality")}'
            f'{stat("Off-peak util", f"{m.offpeak_utilization*100:.0f}%")}'
            f'</div>')
    display(HTML(html))

## 6. Inspect a deal — bilateral negotiation thread

Pick any deal from the run above. Below the deal summary, you'll see the *implicit conversation* between the buyer and the seller's slot — only the offers from those two parties, in round order, like a 1:1 chat thread.

The full marketplace tape (everyone shouting their offers) is in cell 6.6. **This view is the bilateral subset** — the negotiation that actually concluded the deal.

In live LLM mode the reasoning text on each bubble is genuine model output, so the conversation reads more like an actual back-and-forth ("I'm matching your $3.40 ask because the round-4 deadline is approaching" etc.). In fast mode it's deterministic stub text.

In [ ]:
from gpubid.viz.trace_view import render_trace

if not final.all_deals:
    display(HTML('<em>No deals to inspect — try a slack-supply market or more rounds.</em>'))
else:
    deal_options = [(f'{d.buyer_id} ↔ {d.seller_id} · {d.gpu_type.value}×{d.qty} · ${d.price_per_gpu_hr:.2f}/hr (round {d.round_n})', d.id) for d in final.all_deals]
    deal_select = w.Dropdown(options=deal_options, description='Deal')

    out = w.Output()
    def _on_change(change):
        deal = next(d for d in final.all_deals if d.id == change['new'])
        out.clear_output(wait=True)
        with out:
            display(HTML(render_trace(deal, market, snapshots=all_snapshots)))
    deal_select.observe(_on_change, names='value')
    # Initial render — passes all_snapshots so the bilateral negotiation thread
    # (just this buyer ↔ this seller's slot, in chronological order) renders.
    with out:
        display(HTML(render_trace(final.all_deals[0], market, snapshots=all_snapshots)))
    display(deal_select, out)

## 6.5 Negotiation forensics — who held out, who conceded?

The animation in cell 3 shows offers in flight; this view reconstructs the round-by-round actions every agent took, with two diagnostics:

- **Timeline chart**: each agent's posted price across rounds. Solid line = won a deal; dashed = held out. Stars mark where deals cleared.
- **Aggression scoreboard**: how much each agent moved from their first offer to their last. Buyers in `+%` climbed (raised bids); sellers in `+%` decayed (dropped asks).

Below those: a per-round log of every action with ↑/↓ arrows showing direction.

In [ ]:
from gpubid.analysis.forensics import (
    extract_history, render_timeline, render_aggression_scoreboard, render_log,
)

# Reuse `all_snapshots` from cell 3 (the animation cell). In live mode this avoids
# re-paying for the LLM calls; in fast mode it's just faster.
history = extract_history(market, all_snapshots)

# 1) Offer trajectories
render_timeline(history).show()

# 2) Aggression scoreboard
display(HTML(render_aggression_scoreboard(history)))

# 3) Round-by-round log (collapsed by default, scroll to read)
display(HTML(
    '<details><summary style="cursor:pointer;font-size:13px;color:#374151;font-weight:600;">'
    'Round-by-round log (click to expand)</summary>' + render_log(history) + '</details>'
))

## 6.6 Chat exchange — what each agent actually said

The most interesting part of the negotiation: **the reasoning text the agents wrote**. In live LLM mode this is genuine model output — the agent's stated strategy, hedging, second-guessing. In fast mode the deterministic agents just stamp something like `"opening ask at $5.40"`, but the same renderer works for both.

Bubbles aligned right are buyers; left are sellers. Color-coded by side. Toggle `only_with_reasoning=True` if you want to skip the deterministic stubs and only see LLM-written messages.

In [ ]:
from gpubid.analysis.forensics import render_chat_exchange
from gpubid.engine.round_runner import (
    make_deterministic_agents, make_llm_agents, agent_models_map,
)

# Reconstruct the agent dicts so we can look up which provider/model each agent used.
# This mirrors what cell 10 (animate) used.
if mode_label.startswith('fast') or mode_label.startswith('preset'):
    _b_agents, _s_agents = make_deterministic_agents(market)
elif mode_label.startswith('live'):
    _b_agents, _s_agents = make_llm_agents(market, api_key=api_key_input.value)
else:
    _b_agents, _s_agents = {}, {}

models_map = agent_models_map(_b_agents, _s_agents)
display(HTML(render_chat_exchange(history, only_with_reasoning=False,
                                  max_height_px=600, agent_models=models_map)))

## 6.65 Bilateral negotiation dialogue (real back-and-forth)

The cell above (6.6) shows the marketplace tape — every agent broadcasting to the public board. **This cell is different**: for each closed deal, we run a true 1:1 negotiation between just that buyer and that seller, with strategic prompts that demand:

- **Arguments with reasons** (capacity scarcity, alternatives, deadline pressure)
- **Conditional concessions** ("I\'ll go to $X if you commit to N hours")
- **References to alternatives** ("another seller is at $Z" / "your fallback is posted-price")
- **Walk-away threats** when one side won\'t move

The model badges on each bubble show which provider/model is on each side, so the conversation experience makes clear "Claude Haiku just argued the seller down by citing OpenAI gpt-4o-mini\'s prior counter."

⚠️ **This cell makes real LLM calls** — only runs in `live` mode. Cost is roughly 2-6 LLM calls per dialogue × your number of deals. Skip if you\'re not in live mode.

In [ ]:
from gpubid.protocol.dialogue import run_bilateral_dialogue
from gpubid.viz.dialogue_view import render_dialogue

if mode_label != 'live':
    display(HTML(
        '<em>Bilateral dialogue requires live mode (real LLM calls per turn). '
        'Switch the Mode dropdown to "live" and re-run cell 13.</em>'
    ))
elif not final.all_deals:
    display(HTML('<em>No deals to dialogue over — try a slack-supply market or more rounds.</em>'))
else:
    from gpubid.llm import make_client
    client = make_client(api_key_input.value)

    # Up to 3 deals per run to keep cost bounded (~$0.05-0.15 per run).
    deals_to_replay = list(final.all_deals)[:3]
    print(f'Running bilateral dialogues for {len(deals_to_replay)} deals (~$0.05-0.15 of API calls)...')

    for d in deals_to_replay:
        buyer  = next(b for b in market.buyers  if b.id == d.buyer_id)
        seller = next(s for s in market.sellers if s.id == d.seller_id)
        slot   = next(sl for sl in seller.capacity_slots if sl.id == d.slot_id)

        # v0.3 enrichment: pass the seller's volume policy + buyer's CEO brief
        # into the dialogue prompts so agents can argue about volume commits and
        # ground their reasoning in the buyer's actual business context.
        seller_policy = enrichment.seller_volume_policies.get(seller.id)
        buyer_priv = enrichment.buyer_private_profiles.get(buyer.id)
        biz_context = buyer_priv.business_context_summary if buyer_priv else None

        b_open = buyer.job.max_value_per_gpu_hr * 0.6
        s_open = slot.reserve_per_gpu_hr * 1.5

        result = run_bilateral_dialogue(
            buyer=buyer, seller=seller, slot=slot,
            opening_seller_price=s_open,
            opening_buyer_price=b_open,
            max_turns=6,
            buyer_client=client,
            seller_client=client,
            market=market,
            posted_price_estimate=slot.reserve_per_gpu_hr * 1.4,
            seller_volume_policy=seller_policy,
            buyer_business_context=biz_context,
        )
        display(HTML(render_dialogue(result, max_height_px=520)))

## 6.7 Provider tournament — Claude vs OpenAI vs Gemini

Run the same market under different LLM-provider assignments to answer:

- **Who wins more deals — Claude, OpenAI, or Gemini buyers?**
- **Are some providers more aggressive at decaying prices than others?**
- **Does the welfare-vs-collusion gap show up at this sample size?**

Configure `PROVIDERS` below to choose any subset of `{anthropic, openai, gemini}`. Round-robin assignment splits buyers (and sellers) evenly across the providers you select.

⚠️ **Real LLM calls cost API credits.** 3 providers × 5 seeds × 12 agents × 5 rounds ≈ 900 calls (~\$1-3 with cost-effective default models). The default below uses the deterministic smoke test (no keys needed) so you can verify the framework before spending — flip `MODE = "live"` to run the real comparison.

Right after the tournament, **cell 6.75** runs the persuasion analytics on the same seeds — that's where you see *which* provider bluffs more, which makes the most concessions, and so on.

In [ ]:
import os
from gpubid.analysis.tournament import (
    head_to_head_deterministic,
    head_to_head_alternating,
    head_to_head_multi,
    render_tournament_report,
    render_tournament_chart,
    compute_baseline_comparison,
    render_baseline_comparison,
)

# Configurable knobs.
MODE = "deterministic"   # "deterministic" | "live"  (live needs API keys)
PROVIDERS = ["anthropic", "openai", "gemini"]   # any subset of 2-3
N_SEEDS = 5
N_BUYERS = 8
N_SELLERS = 4
REGIME = "tight"
MAX_ROUNDS = 5

# Optional model pinning per provider (set to None for cost-effective defaults).
PROVIDER_MODELS = {
    "anthropic": None,           # default: claude-haiku-4-5
    "openai":    None,           # default: gpt-4o-mini
    "gemini":    None,           # default: gemini-2.5-flash
    # to swap in pricier/smarter models, e.g.
    #   "anthropic": "claude-sonnet-4-6",
    #   "openai":    "gpt-4o",
    #   "gemini":    "gemini-2.5-pro",
}

if MODE == "live":
    api_keys = {}
    if 'google.colab' in sys.modules:
        try:
            from google.colab import userdata
            for prov, name in [("anthropic", "ANTHROPIC_API_KEY"),
                               ("openai",    "OPENAI_API_KEY"),
                               ("gemini",    "GEMINI_API_KEY")]:
                try:
                    v = userdata.get(name)
                    if v: api_keys[prov] = v
                except Exception:
                    pass
        except ImportError:
            pass
    # Env-var fallback (for local) or password-field key (for Colab manual entry).
    for prov, env in [("anthropic", "ANTHROPIC_API_KEY"),
                      ("openai",    "OPENAI_API_KEY"),
                      ("gemini",    "GEMINI_API_KEY")]:
        v = os.environ.get(env)
        if v: api_keys.setdefault(prov, v)

    # Drop any provider for which we couldn't find a key.
    runnable_providers = [p for p in PROVIDERS if api_keys.get(p)]
    missing = [p for p in PROVIDERS if not api_keys.get(p)]
    if missing:
        print(f'⚠ no key for {missing} — skipping those. Running with: {runnable_providers}')

    if len(runnable_providers) < 2:
        raise RuntimeError(
            f'Live tournament needs >=2 provider keys. Found: {runnable_providers}. '
            'Set ANTHROPIC_API_KEY / OPENAI_API_KEY / GEMINI_API_KEY in Colab Secrets.'
        )

    pinned = {p: PROVIDER_MODELS.get(p) for p in runnable_providers if PROVIDER_MODELS.get(p)}
    result = head_to_head_multi(
        n_seeds=N_SEEDS,
        api_keys={p: api_keys[p] for p in runnable_providers},
        n_buyers=N_BUYERS, n_sellers=N_SELLERS, regime=REGIME, max_rounds=MAX_ROUNDS,
        provider_models=pinned or None,
    )
else:
    result = head_to_head_deterministic(
        n_seeds=N_SEEDS, n_buyers=N_BUYERS, n_sellers=N_SELLERS,
        regime=REGIME, max_rounds=MAX_ROUNDS,
    )

display(HTML(render_tournament_report(result)))
render_tournament_chart(result).show()

# Baseline comparison: how does each seed's agentic outcome compare to VCG / posted?
baseline_rows = compute_baseline_comparison(result)
display(HTML(render_baseline_comparison(baseline_rows)))

## 6.75 Persuasion + manipulation analytics

The tournament above tells you who *won*; this cell tells you *how*. Two complementary views:

1. **Quantitative persuasion** — for each agent, the average % the counterparty's posted price moved *after* this agent posted. Buyers who pull sellers DOWN get a positive score; sellers who pull buyers UP get a positive score. No LLM needed.
2. **Semantic style** — every reasoning bubble in the run is sent to a separate LLM **judge** that tags it with one or more of: `bluff`, `false_urgency`, `emotional_appeal`, `anchor`, `concession`, `honest_argument`, `hedge`. We then aggregate by agent and by provider — this is how we answer "does Claude bluff more than Gemini?"

The judge runs in `live` mode only (requires an API key) and costs roughly \$0.05-0.10 per market run. In `preset`/`deterministic` mode you still get the quantitative score; the tag columns will be empty.

Toggle `JUDGE_PROVIDER` below to pick which model is the umpire. Using a *different* provider as the judge than the agents helps avoid one model rating itself favorably.

In [ ]:
from gpubid.analysis.persuasion import (
    analyze_persuasion,
    render_persuasion_summary,
    render_persuasion_examples,
    render_persuasion_chart,
)
from gpubid.engine.round_runner import agent_models_map
from gpubid.llm import make_client

# Pick a judge: which LLM rates the reasoning bubbles? Set to None to skip the
# semantic tags (you'll still get quantitative persuasion). Setting it to the
# pasted-in api_key reuses whatever you set up in cell 4.
JUDGE_KEY = api_key_input.value if mode_label == 'live' else None
JUDGE_MODEL = None    # e.g. "claude-haiku-4-5-20251001" / "gpt-4o-mini" / "gemini-2.5-flash"
MAX_BUBBLES_TO_JUDGE = 80   # caps cost per run; bump to 200 for deeper analysis

judge = None
if JUDGE_KEY:
    try:
        judge = make_client(JUDGE_KEY, model=JUDGE_MODEL)
        print(f'Persuasion judge: {judge.provider}/{judge.model}')
    except Exception as e:
        print(f'Could not init judge: {e}. Skipping semantic tags — quantitative scores only.')

# Reuse the agent-models map from cell 6.6 if available, else rebuild it.
try:
    persuasion_models_map = models_map
except NameError:
    from gpubid.engine.round_runner import make_deterministic_agents, make_llm_agents
    if mode_label == 'live' and api_key_input.value:
        _b, _s = make_llm_agents(market, api_key=api_key_input.value)
    else:
        _b, _s = make_deterministic_agents(market)
    persuasion_models_map = agent_models_map(_b, _s)

persuasion_report = analyze_persuasion(
    history,
    judge=judge,
    agent_models=persuasion_models_map,
    max_bubbles=MAX_BUBBLES_TO_JUDGE,
)

display(HTML(render_persuasion_summary(persuasion_report)))

# Plotly chart: per-provider tag mix
if judge is not None and any(ag.tag_counts for ag in persuasion_report.agents.values()):
    render_persuasion_chart(persuasion_report).show()
    display(HTML('<details open><summary style="cursor:pointer;font-weight:600;color:#374151;">'
                 'Example utterances by tag</summary>'
                 + render_persuasion_examples(persuasion_report) + '</details>'))
else:
    display(HTML(
        '<div style="font-size:12px;color:#6b7280;font-style:italic;margin-top:6px;">'
        'No semantic tags scored — switch Mode to <code>live</code> with a valid API key '
        'and re-run cell 13 + this cell to enable the judge.'
        '</div>'
    ))

## 6.8 Post-deal regret signals (exploratory)

After each deal closes, we synthesize a **regret signal** for both sides — buyers regret overpaying, sellers regret leaving money on the table. In a real marketplace this comes from post-deal user feedback; here we generate it from the private profiles to demonstrate the framing.

The demo of correction is simple but useful: if seller agents systematically show high regret, inject a calibration line into their next-run system prompt (`"Recent runs suggest you have been accepting prices close to your reserve. Hold firmer."`). This is **prompt-time calibration based on signals**, not a real RLHF loop.

This is a "future work" framing for the class discussion: the marketplace platform itself can collect feedback signals and recalibrate participant agents over time.

In [ ]:
from gpubid.analysis.regret import synthesize_regret, calibration_line_for_seller
from gpubid.market_v3 import V3Enrichment

# We need v0.3 buyer/seller profiles for synthesize_regret. Build them from the
# enrichment populated in cell 11.
buyer_v3_lookup = {}
for buyer in market.buyers:
    pub = enrichment.buyer_public_profiles.get(buyer.id)
    priv = enrichment.buyer_private_profiles.get(buyer.id)
    if pub and priv:
        from gpubid.domain.profiles import BuyerV2
        buyer_v3_lookup[buyer.id] = BuyerV2(public=pub, private=priv)

seller_v3_lookup = enrichment.seller_v3

if not final.all_deals or not buyer_v3_lookup or not seller_v3_lookup:
    display(HTML('<em>Regret view requires v0.3 enrichment + at least one closed deal. Run cell 13 first.</em>'))
else:
    rows = []
    seller_regret_scores = []
    for d in final.all_deals:
        buyer = buyer_v3_lookup.get(d.buyer_id)
        seller = seller_v3_lookup.get(d.seller_id)
        if buyer is None or seller is None:
            continue
        signal = synthesize_regret(d, buyer, seller)
        seller_regret_scores.append(signal.seller_score)
        # Render row
        b_color = '#dc2626' if signal.buyer_score > 0.7 else '#f59e0b' if signal.buyer_score > 0.4 else '#16a34a'
        s_color = '#dc2626' if signal.seller_score > 0.7 else '#f59e0b' if signal.seller_score > 0.4 else '#16a34a'
        rows.append(
            f'<tr>'
            f'<td style="padding:6px 10px;font-family:monospace;font-size:11px;">{d.id}</td>'
            f'<td style="padding:6px 10px;font-size:11px;">{d.buyer_id}<span style="color:#9ca3af;"> ↔ </span>{d.seller_id}</td>'
            f'<td style="padding:6px 10px;font-family:monospace;font-size:11px;">\u0024{d.price_per_gpu_hr:.2f}/hr</td>'
            f'<td style="padding:6px 10px;color:{b_color};font-family:monospace;font-weight:600;">{signal.buyer_score:.2f}</td>'
            f'<td style="padding:6px 10px;color:{s_color};font-family:monospace;font-weight:600;">{signal.seller_score:.2f}</td>'
            f'<td style="padding:6px 10px;font-size:11px;color:#6b7280;font-style:italic;">{signal.notes or "no flags"}</td>'
            f'</tr>'
        )

    avg_seller_regret = sum(seller_regret_scores) / len(seller_regret_scores) if seller_regret_scores else 0
    cal_line = calibration_line_for_seller(avg_seller_regret)
    cal_block = (
        f'<div style="margin-top:12px;padding:10px;background:#fef3c7;border-left:4px solid #f59e0b;border-radius:4px;">'
        f'<strong>Calibration trigger:</strong> avg seller regret = {avg_seller_regret:.2f}. '
        f'Inject this line into next-run seller prompts:<br>'
        f'<em>\u201c{cal_line}\u201d</em></div>'
    ) if cal_line else (
        f'<div style="margin-top:12px;padding:10px;background:#dcfce7;border-left:4px solid #16a34a;border-radius:4px;">'
        f'<strong>No calibration needed</strong> — avg seller regret = {avg_seller_regret:.2f} (below 0.4 threshold).'
        f'</div>'
    )

    display(HTML(
        '<table style="border-collapse:collapse;font-family:-apple-system,sans-serif;width:100%;max-width:900px;">'
        '<thead><tr style="background:#f3f4f6;text-align:left;font-size:11px;color:#374151;">'
        '<th style="padding:6px 10px;border-bottom:1px solid #d1d5db;">Deal</th>'
        '<th style="padding:6px 10px;border-bottom:1px solid #d1d5db;">Pair</th>'
        '<th style="padding:6px 10px;border-bottom:1px solid #d1d5db;">Price</th>'
        '<th style="padding:6px 10px;border-bottom:1px solid #d1d5db;">Buyer regret</th>'
        '<th style="padding:6px 10px;border-bottom:1px solid #d1d5db;">Seller regret</th>'
        '<th style="padding:6px 10px;border-bottom:1px solid #d1d5db;">Notes</th>'
        '</tr></thead><tbody>'
        + ''.join(rows) + '</tbody></table>'
        + cal_block
    ))

## 6.9 Human-in-the-loop trigger demo

In a production marketplace platform, certain situations should escalate to a human:

- **Constraint violation imminent**: an agent\'s next offer would cross its private cap (buyer overpaying, seller undercutting reserve).
- **Deadlock**: a pair has accumulated multiple no-progress rounds; the platform asks a human whether to keep paying for tokens or break the deadlock.
- **Low-confidence close**: an agent emits low confidence on an `accept` action.
- **Ambiguous requirement**: the buyer\'s NL requirement was so vague that the translate step yields a wide qty/duration range.

Below is a stub demonstration. The notebook\'s headless surfacer auto-proceeds on every trigger (so this cell runs without a widget); a real demo with `ipywidgets.Button` is wired up in `gpubid.protocol.hitl` for the live class showcase.

In [ ]:
from gpubid.protocol.hitl import (
    HITLPolicy, HITLEvent, HITLTrigger, auto_proceed_surfacer,
)
from gpubid.domain.requirements import REQUIREMENT_LIBRARY

# Build a few synthetic events that would have fired in this market run.
events = []
# 1. Ambiguous requirement on buyers whose qty range spans wide
for r in REQUIREMENT_LIBRARY:
    width = r.expected_qty_range[1] - r.expected_qty_range[0]
    midpoint = (r.expected_qty_range[0] + r.expected_qty_range[1]) / 2
    if midpoint > 0 and width / midpoint > 0.5:
        events.append(HITLEvent(
            trigger=HITLTrigger.AMBIGUOUS_REQUIREMENT,
            agent_id=f'B?',
            round_n=0,
            proposed_offer=None,
            note=f'requirement {r.requirement_id}: qty range {r.expected_qty_range} (width/midpoint={width/midpoint:.0%})',
        ))

# 2. Pretend a deadlock fired between two agents (using forensics history)
if final.all_deals:
    sample_deal = final.all_deals[0]
    events.append(HITLEvent(
        trigger=HITLTrigger.LOW_CONFIDENCE_CLOSE,
        agent_id=sample_deal.buyer_id,
        round_n=sample_deal.round_n,
        proposed_offer=None,
        note=f'pretend: buyer {sample_deal.buyer_id} closed deal at \u0024{sample_deal.price_per_gpu_hr:.2f}/hr — would surface for confirmation if confidence were low',
    ))

# Run them through the headless policy
policy = HITLPolicy(enabled=True, surfacer=auto_proceed_surfacer)
log_rows = []
for event in events[:5]:
    decision = policy.maybe_intervene(event)
    color = {
        HITLTrigger.CONSTRAINT_VIOLATION_IMMINENT: '#dc2626',
        HITLTrigger.DEADLOCK: '#d97706',
        HITLTrigger.LOW_CONFIDENCE_CLOSE: '#7c3aed',
        HITLTrigger.AMBIGUOUS_REQUIREMENT: '#0369a1',
    }[event.trigger]
    log_rows.append(
        f'<div style="background:#fff;border-left:4px solid {color};border:1px solid #e5e7eb;'
        f'padding:8px 12px;margin-bottom:6px;font-family:-apple-system,sans-serif;font-size:12px;border-radius:4px;">'
        f'<div style="display:flex;justify-content:space-between;">'
        f'<strong style="color:{color};">⚠️ {event.trigger.value}</strong>'
        f'<span style="font-family:monospace;color:#9ca3af;font-size:10px;">round {event.round_n} · agent {event.agent_id}</span>'
        f'</div>'
        f'<div style="color:#374151;margin-top:4px;">{event.note}</div>'
        f'<div style="color:#6b7280;margin-top:4px;font-size:11px;">→ headless surfacer: <code>{decision.action}</code> ({decision.note})</div>'
        f'</div>'
    )

display(HTML(
    '<div>' + ''.join(log_rows) + '</div>'
    + ('<em style="color:#6b7280;font-size:11px;">No HITL triggers fired in this seed — try a different seed or a tighter token cap.</em>' if not log_rows else '')
))

## 6.95 Where HITL pays off — real-world use cases

The cell above shows the *trigger taxonomy* (what kinds of events should escalate). This cell zooms out: which kinds of GPU-marketplace transactions actually *need* human review, and how the persuasion analytics from 6.75 feed back into automatic escalation.

The seven scenarios below are the cases where 30 seconds of human attention almost always beats fully-autonomous execution: high-stakes financial commitments, regulated workloads, all-or-nothing training runs, and counterparties showing manipulation signals. Each card lists the practical threshold and the auto-detected signal that should trigger HITL.

In [ ]:
from gpubid.analysis.hitl_usecases import (
    render_hitl_use_cases,
    detect_alerts_from_persuasion,
    render_hitl_alerts,
)

# Static guidance: the seven canonical use cases.
display(HTML(render_hitl_use_cases()))

# Live: did THIS run produce any HITL alerts? Pulls from the persuasion report
# in cell 6.75 — if the judge tagged a counterparty as bluff/false_urgency/
# emotional_appeal more than twice, surface their counterparties for review.
try:
    alerts = detect_alerts_from_persuasion(persuasion_report)
    display(HTML(
        '<h4 style="margin:14px 0 6px;font-family:-apple-system,sans-serif;">'
        'Live alerts from this run</h4>'
        + render_hitl_alerts(alerts)
    ))
except NameError:
    display(HTML(
        '<em style="color:#9ca3af;">Run cell 6.75 (persuasion analytics) first to populate live alerts.</em>'
    ))

## 7. Headline figures

Pre-rendered figures from `data/figures/`, regenerable via `PYTHONPATH=src python -m gpubid.experiments.run_sweep`.

These are the four figures that go on the one-pager:
1. **Welfare recovery vs round count** — diminishing returns; N=5 sits on the elbow.
2. **Welfare and Gini vs concentration cap** — fairness vs welfare tradeoff.
3. **Agentic vs posted-price** — across 15+ seeds in both regimes.
4. **Off-peak slot utilization** — agentic fills off-peak; posted-price doesn't.

In [ ]:
from IPython.display import Image, display
from pathlib import Path

figures_dir = Path(REPO_ROOT) / 'data' / 'figures'
for name in ['welfare_vs_rounds.png', 'welfare_vs_cap.png', 'agentic_vs_posted.png', 'offpeak_utilization.png']:
    path = figures_dir / name
    if path.exists():
        display(Image(str(path)))
    else:
        display(HTML(f'<em>Missing {name} — regenerate with:'
                     f' <code>PYTHONPATH=src python -m gpubid.experiments.run_sweep</code></em>'))

## 8. Writeup

**Problem.** GPU cloud markets force buyers into three rigid contract types: long-term reserved, on-demand, or spot. A team that needs 8 H100s in the next hour pays the same on-demand rate as a team willing to wait 12 hours, even though they impose very different claims on perishable capacity. The middle of the demand curve is unserved.

**Mechanism.** Buyer agents and seller agents post structured offer tuples (price, qty, GPU type, time slot, duration, interruption tolerance) plus free-form reasoning. They negotiate over rounds via a public board. A central engine clears agreed deals, enforces a concentration cap, and validates against private reserves.

**Headline result.** Across 15+ seeds and both supply regimes, the agentic mechanism recovers around 90% of the VCG welfare upper bound, beating the posted-price baseline by a wide margin in slack-supply markets where off-peak slots otherwise go unfilled.

**Tradeoffs.** See the proposal: structure vs autonomy (we chose structured tuples + free-form reasoning), welfare vs collusion (heterogeneous backends — provider mix toggle in M5), expressiveness vs tractability (6-dim bids, default 8×4 markets), and fairness vs revenue (20% concentration cap). The first and second require LLM agents to demonstrate; baked presets in `data/presets/` cover them.